In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. POWTARZALNOŚĆ
# ==========================================

np.random.seed(42)
tf.random.set_seed(42)

# ==========================================
# 2. SYNTETYCZNE DANE "SYNAPSE X"
# ==========================================
# 8 cech opisujących obiekt decyzyjny:
# novelty, feasibility, cost_efficiency, stability,
# risk, strategic_alignment, explainability, adaptability
#
# y = 1 oznacza obiekt "wartościowy / perspektywiczny"
# ==========================================

n_samples = 400

X = np.random.rand(n_samples, 8).astype(np.float32)

feature_names = [
    "novelty",
    "feasibility",
    "cost_efficiency",
    "stability",
    "risk",
    "strategic_alignment",
    "explainability",
    "adaptability"
]

# Ukryta reguła generująca etykiety:
# nieco nieliniowa, z premią za wysoką innowacyjność + alignment + adaptability
# i karą za niektóre konfiguracje ryzyka / stabilności

score_hidden = (
    1.8 * X[:, 0] +   # novelty
    1.5 * X[:, 1] +   # feasibility
    1.2 * X[:, 2] +   # cost_efficiency
    1.3 * X[:, 5] +   # strategic_alignment
    1.4 * X[:, 7] -   # adaptability
    1.1 * X[:, 4] +   # risk
    0.8 * X[:, 3]     # stability
)

# dodatkowa nieliniowość
nonlinear_bonus = (
    0.8 * (X[:, 0] * X[:, 5]) +
    0.7 * (X[:, 1] * X[:, 7]) -
    0.9 * (X[:, 4] * X[:, 3])
)

score_hidden = score_hidden + nonlinear_bonus
threshold = np.median(score_hidden)
y = (score_hidden > threshold).astype(np.int32)

df = pd.DataFrame(X, columns=feature_names)
df["label"] = y

print("Przykładowe dane:")
print(df.head())

# ==========================================
# 3. BANK ATRAKTORÓW
# ==========================================
# Każdy atraktor reprezentuje inny archetyp "dobrego rozwiązania"
# ==========================================

attractors = {
    "visionary": np.array([0.95, 0.70, 0.60, 0.45, 0.65, 0.95, 0.55, 0.90], dtype=np.float32),
    "balanced":  np.array([0.70, 0.80, 0.75, 0.80, 0.35, 0.78, 0.80, 0.70], dtype=np.float32),
    "robust":    np.array([0.55, 0.90, 0.85, 0.92, 0.20, 0.82, 0.90, 0.65], dtype=np.float32),
}

attractor_names = list(attractors.keys())

# opcjonalne wagi cech dla rezonansu
feature_weights = np.array([1.4, 1.3, 1.0, 1.2, 1.5, 1.4, 1.1, 1.2], dtype=np.float32)

def weighted_distance_matrix(X, attractors_dict, weights):
    """
    Zwraca macierz odległości: [n_samples, n_attractors]
    """
    distances = []
    for name in attractor_names:
        a = attractors_dict[name]
        diff = X - a
        dist = np.sqrt(np.sum(weights * diff**2, axis=1))
        distances.append(dist)
    return np.stack(distances, axis=1)

def resonance_from_distances(dist_matrix):
    """
    Zamiana odległości na rezonanse.
    """
    return 1.0 / (1.0 + dist_matrix)

dist_matrix = weighted_distance_matrix(X, attractors, feature_weights)
res_matrix = resonance_from_distances(dist_matrix)

# dodatkowe cechy meta
best_resonance = np.max(res_matrix, axis=1, keepdims=True)
best_attractor_idx = np.argmax(res_matrix, axis=1, keepdims=True)

# niepewność atraktorowa:
# jeśli dwa najlepsze rezonanse są podobne, system jest mniej pewny
sorted_res = np.sort(res_matrix, axis=1)
uncertainty_attr = 1.0 - (sorted_res[:, -1] - sorted_res[:, -2]).reshape(-1, 1)

# X_aug = surowe cechy + rezonanse + meta-cechy
X_aug = np.hstack([
    X,
    res_matrix,
    best_resonance,
    uncertainty_attr
]).astype(np.float32)

aug_feature_names = (
    feature_names +
    [f"res_{name}" for name in attractor_names] +
    ["best_resonance", "uncertainty_attr"]
)

print("\nRozszerzone cechy:")
print(pd.DataFrame(X_aug, columns=aug_feature_names).head())

# ==========================================
# 4. TRAIN / TEST SPLIT + SKALOWANIE
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

# rozdzielamy też surową część i część atraktorową
n_raw = len(feature_names)
n_attr = X_aug.shape[1] - n_raw

X_train_raw = X_train_scaled[:, :n_raw]
X_test_raw = X_test_scaled[:, :n_raw]

X_train_attr = X_train_scaled[:, n_raw:]
X_test_attr = X_test_scaled[:, n_raw:]

# ==========================================
# 5. GAŁĄŹ KLASYCZNA: RANDOM FOREST
# ==========================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]
rf_pred_class = (rf_pred_proba > 0.5).astype(int)

# ==========================================
# 6. GAŁĄŹ NEURALNA: DWUWEJŚCIOWY MODEL SYNAPSE X
# ==========================================
# input 1: surowe cechy
# input 2: cechy atraktorowe
#
# model ma wewnętrzny gating:
# osobna mała warstwa uczy się, jak mocno ufać gałęzi atraktorowej
# ==========================================

raw_input = layers.Input(shape=(n_raw,), name="raw_input")
attr_input = layers.Input(shape=(n_attr,), name="attr_input")

# gałąź surowa
x_raw = layers.Dense(32, activation="relu")(raw_input)
x_raw = layers.Dense(16, activation="relu")(x_raw)

# gałąź atraktorowa
x_attr = layers.Dense(16, activation="relu")(attr_input)
x_attr = layers.Dense(8, activation="relu")(x_attr)

# gating: ile zaufać gałęzi atraktorowej
gate = layers.Dense(8, activation="sigmoid", name="gate_layer")(x_attr)

# modulacja gałęzi atraktorowej
gated_attr = layers.Multiply()([x_attr, gate])

# fuzja
fusion = layers.Concatenate()([x_raw, gated_attr])
fusion = layers.Dense(16, activation="relu")(fusion)
fusion = layers.Dropout(0.15)(fusion)

# embedding synapse
embedding = layers.Dense(6, activation="linear", name="synapse_embedding")(fusion)

# wyjście końcowe
output = layers.Dense(1, activation="sigmoid", name="decision")(embedding)

nn_model = Model(inputs=[raw_input, attr_input], outputs=[output, embedding])

# ==========================================
# 7. ATRAKTOR W PRZESTRZENI UKRYTEJ
# ==========================================
# To jest drugi poziom atrakcji:
# nie tylko w przestrzeni wejścia, ale też w latent space
# ==========================================

latent_attractor = tf.constant([[1.0, 0.8, 0.6, 0.4, 0.9, 0.7]], dtype=tf.float32)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

X_train_raw_tf = tf.convert_to_tensor(X_train_raw, dtype=tf.float32)
X_train_attr_tf = tf.convert_to_tensor(X_train_attr, dtype=tf.float32)
y_train_tf = tf.convert_to_tensor(y_train.reshape(-1, 1), dtype=tf.float32)

X_test_raw_tf = tf.convert_to_tensor(X_test_raw, dtype=tf.float32)
X_test_attr_tf = tf.convert_to_tensor(X_test_attr, dtype=tf.float32)

# ==========================================
# 8. UCZENIE MODELU Z PODWÓJNĄ STRATĄ
# ==========================================
# loss_total = classification_loss + lambda * latent_attractor_loss
# ==========================================

lambda_attr = 0.08
epochs = 120

for epoch in range(epochs):
    with tf.GradientTape() as tape:
        pred_train, emb_train = nn_model([X_train_raw_tf, X_train_attr_tf], training=True)

        classification_loss = tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(y_train_tf, pred_train)
        )

        # atraktor latentny aktywny mocniej dla klasy pozytywnej
        positive_mask = y_train_tf
        latent_diff = emb_train - latent_attractor
        latent_dist = tf.reduce_sum(latent_diff ** 2, axis=1, keepdims=True)

        latent_attr_loss = tf.reduce_mean(positive_mask * latent_dist)

        total_loss = classification_loss + lambda_attr * latent_attr_loss

    grads = tape.gradient(total_loss, nn_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, nn_model.trainable_variables))

    if epoch % 20 == 0:
        print(
            f"epoch={epoch:03d} | "
            f"class_loss={classification_loss.numpy():.4f} | "
            f"latent_attr_loss={latent_attr_loss.numpy():.4f} | "
            f"total_loss={total_loss.numpy():.4f}"
        )

# ==========================================
# 9. PREDYKCJE SIECI
# ==========================================

nn_pred_proba, nn_test_emb = nn_model([X_test_raw_tf, X_test_attr_tf], training=False)
nn_pred_proba = nn_pred_proba.numpy().flatten()
nn_pred_class = (nn_pred_proba > 0.5).astype(int)

# ==========================================
# 10. META-FUZJA: ADAPTACYJNE ŁĄCZENIE GŁOSÓW
# ==========================================
# łączymy:
# - nn_pred_proba
# - rf_pred_proba
# - best_resonance
# - uncertainty_attr
#
# jeśli atraktor jest bardzo pewny, rośnie jego udział
# jeśli sieć i RF są zgodne, rośnie ich wspólna siła
# ==========================================

# odczyt meta-cech na teście
test_best_resonance = X_test[:, -2]
test_uncertainty_attr = X_test[:, -1]

# pewność NN
nn_conf = np.abs(nn_pred_proba - 0.5) * 2.0
rf_conf = np.abs(rf_pred_proba - 0.5) * 2.0

# zgodność modeli
agreement = 1.0 - np.abs(nn_pred_proba - rf_pred_proba)

# dynamiczne wagi
w_nn = 0.35 + 0.25 * nn_conf + 0.10 * agreement
w_rf = 0.30 + 0.20 * rf_conf + 0.10 * agreement
w_attr = 0.20 + 0.35 * test_best_resonance - 0.20 * test_uncertainty_attr

# normalizacja
w_sum = w_nn + w_rf + w_attr
w_nn /= w_sum
w_rf /= w_sum
w_attr /= w_sum

final_score = (
    w_nn * nn_pred_proba +
    w_rf * rf_pred_proba +
    w_attr * test_best_resonance
)

final_pred = (final_score > 0.5).astype(int)

# ==========================================
# 11. RAPORT
# ==========================================

print("\n=== RANDOM FOREST ===")
print("Accuracy:", accuracy_score(y_test, rf_pred_class))
print(classification_report(y_test, rf_pred_class))

print("\n=== NEURAL BRANCH ===")
print("Accuracy:", accuracy_score(y_test, nn_pred_class))
print(classification_report(y_test, nn_pred_class))

print("\n=== SYNAPSE X FUSION ===")
print("Accuracy:", accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred))

# ==========================================
# 12. PODGLĄD WYNIKÓW
# ==========================================

results = pd.DataFrame({
    "y_true": y_test,
    "rf_pred_proba": rf_pred_proba,
    "nn_pred_proba": nn_pred_proba,
    "best_resonance": test_best_resonance,
    "uncertainty_attr": test_uncertainty_attr,
    "w_nn": w_nn,
    "w_rf": w_rf,
    "w_attr": w_attr,
    "final_score": final_score,
    "final_pred": final_pred
})

print("\nPrzykładowe wyniki końcowe:")
print(results.head(20))

Przykładowe dane:
    novelty  feasibility  cost_efficiency  stability      risk  \
0  0.374540     0.950714         0.731994   0.598659  0.156019   
1  0.601115     0.708073         0.020584   0.969910  0.832443   
2  0.304242     0.524756         0.431945   0.291229  0.611853   
3  0.456070     0.785176         0.199674   0.514234  0.592415   
4  0.065052     0.948886         0.965632   0.808397  0.304614   

   strategic_alignment  explainability  adaptability  label  
0             0.155995        0.058084      0.866176      1  
1             0.212339        0.181825      0.183405      0  
2             0.139494        0.292145      0.366362      0  
3             0.046450        0.607545      0.170524      0  
4             0.097672        0.684233      0.440152      1  

Rozszerzone cechy:
    novelty  feasibility  cost_efficiency  stability      risk  \
0  0.374540     0.950714         0.731994   0.598659  0.156019   
1  0.601115     0.708073         0.020584   0.969910  0.83244